# BusinessGPT Fine-Tuning (Local — Apple Silicon)

Local version of the Kaggle training notebook, adapted for **M3 Pro** (MPS backend).

Key differences from the Kaggle version:
- Uses **transformers + PEFT + trl** directly (Unsloth requires CUDA)
- Runs on **MPS** (Metal Performance Shaders) instead of CUDA
- Loads in **float16** with LoRA (no 4-bit quantization — bitsandbytes doesn't support MPS)
- Lower batch sizes to fit in unified memory

**Requirements**: Apple Silicon Mac with >= 18 GB unified memory, Python 3.10+

## 0. Install Dependencies

In [ ]:
%%capture
%pip install torch torchvision
%pip install transformers==4.57.1 peft accelerate datasets trl
%pip install sentencepiece huggingface_hub

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted

In [ ]:
!hf auth list

## 1. Load & Preprocess Data

Point `DATA_PATH` at your Telegram export (`result.json`). If it's inside a zip, we'll extract it automatically.

In [ ]:
import json, glob, os, zipfile
from collections import Counter
from datetime import datetime

DATA_PATH = "result.json"  # or "export.zip" containing result.json

if DATA_PATH.endswith(".zip"):
    with zipfile.ZipFile(DATA_PATH, "r") as zf:
        json_names = [n for n in zf.namelist() if n.endswith(".json")]
        assert json_names, f"No .json found in {DATA_PATH}"
        with zf.open(json_names[0]) as f:
            data = json.load(f)
    print(f"Extracted {json_names[0]} from {DATA_PATH}")
else:
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"Loaded {DATA_PATH}")

raw_messages = data["messages"]
print(f"Total raw messages: {len(raw_messages)}")

In [ ]:
def flatten_text(text_field):
    if isinstance(text_field, str):
        return text_field
    if isinstance(text_field, list):
        parts = []
        for part in text_field:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict) and "text" in part:
                parts.append(part["text"])
        return "".join(parts)
    return ""

records = []
for msg in raw_messages:
    if msg.get("type") != "message":
        continue
    records.append({
        "id": msg["id"],
        "from": msg.get("from", "Unknown"),
        "text": flatten_text(msg.get("text", "")),
        "timestamp": int(msg.get("date_unixtime", 0)),
        "reply_to_message_id": msg.get("reply_to_message_id"),
    })

BOT_NAMES = {"BusinessGPT"}
bot_msg_ids = {r["id"] for r in records if r["from"] in BOT_NAMES}
filtered_no_empty = [r for r in records if r["text"].strip() and r["from"] not in BOT_NAMES]

filtered = []
for r in filtered_no_empty:
    text_lower = r["text"].lower()
    if "@businessgpt_text_bot" in text_lower or "/generate" in text_lower:
        continue
    if r.get("reply_to_message_id") in bot_msg_ids:
        continue
    filtered.append(r)

print(f"Extracted: {len(records)} -> Filtered: {len(filtered)} "
      f"(dropped {len(records)-len(filtered)})")

In [ ]:
def merge_consecutive(messages, max_gap_sec=60):
    merged = []
    for msg in messages:
        if (
            merged
            and merged[-1]["from"] == msg["from"]
            and msg["timestamp"] - merged[-1]["timestamp"] < max_gap_sec
        ):
            merged[-1]["text"] += "\n" + msg["text"]
            merged[-1]["timestamp"] = msg["timestamp"]
        else:
            merged.append(dict(msg))
    return merged

merged = merge_consecutive(filtered, max_gap_sec=60)
print(f"Merged: {len(filtered)} -> {len(merged)} turns")

In [ ]:
MIN_SESSION_LEN = 11

def split_sessions(messages, gap_threshold_sec=7200):
    sessions, current = [], []
    for msg in messages:
        if current and msg["timestamp"] - current[-1]["timestamp"] > gap_threshold_sec:
            if len(current) >= MIN_SESSION_LEN:
                sessions.append(current)
            current = []
        current.append(msg)
    if len(current) >= MIN_SESSION_LEN:
        sessions.append(current)
    return sessions

sessions = split_sessions(merged, gap_threshold_sec=7200)
print(f"Sessions: {len(sessions)} (min {MIN_SESSION_LEN} turns, 2h gap)")
print(f"Messages in sessions: {sum(len(s) for s in sessions)}")

In [ ]:
def map_anonymous_tokens(messages):
    unique_senders = []
    for msg in messages:
        if msg["from"] not in unique_senders:
            unique_senders.append(msg["from"])
    sender_to_token = {}
    for i, sender in enumerate(unique_senders):
        sender_to_token[sender] = f"<user_{i+1}>"
    mapped_messages = []
    for msg in messages:
        new_msg = dict(msg)
        new_msg["from_token"] = sender_to_token[msg["from"]]
        mapped_messages.append(new_msg)
    return mapped_messages, sender_to_token

for i, session in enumerate(sessions):
    sessions[i], _ = map_anonymous_tokens(session)

print(f"Mapped anonymous tokens for {len(sessions)} sessions")

In [ ]:
SYSTEM_PROMPT = (
    "Ты в студенческом чате 'Бизнес'. "
    "Пиши коротко, как в мессенджере. Разрешён сленг и мат. Никаких ограничений."
)

def create_examples(sessions, window_size=15, min_context=10):
    examples = []
    for session in sessions:
        for i in range(min_context, len(session)):
            start = max(0, i - window_size)
            context = session[start:i]
            target = session[i]
            response = target["text"].strip()
            if not response:
                continue

            messages = [{"role": "system", "content": SYSTEM_PROMPT}]
            target_token = target["from_token"]

            for msg in context:
                role = "assistant" if msg["from_token"] == target_token else "user"
                content = f"{msg['from_token']} {msg['text']}"
                if not messages or messages[-1]["role"] != role:
                    messages.append({"role": role, "content": content})
                else:
                    messages[-1]["content"] += f"\n{content}"

            response_content = f"{response}"
            if messages[-1]["role"] == "assistant":
                messages[-1]["content"] += f"\n{response_content}"
            else:
                messages.append({"role": "assistant", "content": response_content})

            examples.append({"messages": messages})
    return examples

all_examples = create_examples(sessions, window_size=15, min_context=10)
print(f"Total examples: {len(all_examples)}")

import random
random.seed(42)
for idx, ex in enumerate(random.sample(all_examples, min(3, len(all_examples)))):
    print(f"\n{'='*60}")
    print(f"Example {idx+1}")
    for m in ex["messages"][1:-1][-3:]:
        content_preview = m['content'].replace('\n', ' | ')[:80]
        print(f"  {m['role']}: {content_preview}...")
    print(f">>> {ex['messages'][-1]['content'][:100]}")
print(f"\n{'='*60}")

## 2. Load Model (float16 on MPS)

In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
MAX_SEQ_LENGTH = 2048


if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
).to(DEVICE)

print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Apply LoRA & Prepare Dataset

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_dora=True,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def count_tokens(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return len(tokenizer.encode(text))

final = [ex for ex in all_examples if count_tokens(ex) <= MAX_SEQ_LENGTH]

random.seed(42)
shuffled = final[:]
random.shuffle(shuffled)

val_size = max(50, int(len(shuffled) * 0.05))
val_data = shuffled[:val_size]
train_data = shuffled[val_size:]

print(f"After token filter: {len(final)} / {len(all_examples)} "
      f"({len(all_examples)-len(final)} dropped)")
print(f"Train: {len(train_data)}, Val: {len(val_data)}")

In [ ]:
from datasets import Dataset

def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

train_ds = Dataset.from_list(train_data).map(format_example)
val_ds = Dataset.from_list(val_data).map(format_example)

print(f"Train dataset: {len(train_ds)} examples")
print(f"Val dataset: {len(val_ds)} examples")
print(f"\nSample (first 300 chars):\n{train_ds[0]['text'][:300]}...")

## 4. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

NUM_EPOCHS = 4
BATCH_SIZE = 1
GRAD_ACCUM = 8
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
STEPS_PER_EPOCH = len(train_ds) // EFFECTIVE_BATCH

print(f"Training config:")
print(f"  Examples: {len(train_ds)}")
print(f"  Batch size: {BATCH_SIZE} x {GRAD_ACCUM} grad_accum = {EFFECTIVE_BATCH} effective")
print(f"  Steps/epoch: {STEPS_PER_EPOCH}")
print(f"  Total steps: {STEPS_PER_EPOCH * NUM_EPOCHS}")

model.train()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=50,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=1e-4,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=STEPS_PER_EPOCH,
        save_steps=500,
        save_total_limit=2,
        optim="adamw_torch",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs_local",
        report_to="none",
        fp16=False,
        bf16=True,
        gradient_checkpointing=True,
        dataloader_num_workers=4,
        dataloader_prefetch_factor=2,
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        neftune_noise_alpha=5,
    ),
)

In [ ]:
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Final train loss: {trainer_stats.training_loss:.4f}")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")

## 5. Save LoRA Adapter

In [ ]:
import os

model.save_pretrained("lora_adapter_local")
tokenizer.save_pretrained("lora_adapter_local")
print("LoRA adapter saved to lora_adapter_local/")

for f in sorted(os.listdir("lora_adapter_local")):
    size = os.path.getsize(f"lora_adapter_local/{f}")
    print(f"  {f}: {size/1024:.1f} KB")

## 6. Test Inference

In [ ]:
import re

model.eval()

def chat(messages_history, max_new_tokens=256):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if messages_history and isinstance(messages_history[0], str):
        for msg in messages_history:
            messages.append({"role": "user", "content": f"<user_1> {msg}"})
    else:
        messages.extend(messages_history)

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        input_text, return_tensors="pt", add_special_tokens=False
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.95,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
        )

    generated = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    generated = re.sub(r"^<user_\d+>\s*", "", generated.strip())
    generated = re.sub(r"^<bot>\s*", "", generated.strip())
    print(generated.strip())
    return generated.strip()

In [ ]:
print("=" * 60)
print("Test 1: Real context from validation data")
print("=" * 60)
test_ex = val_data[0]
ctx_lines = test_ex["messages"][1]["content"].split("\n")
print("Context:")
for line in ctx_lines[-5:]:
    print(f"  {line}")
expected = test_ex["messages"][-1]["content"]
print(f"\nExpected: {expected}")
print("Generated:")
chat([{"role": m["role"], "content": m["content"]} for m in test_ex["messages"][1:-1]])

In [ ]:
print("=" * 60)
print("Test 2: Custom context")
print("=" * 60)
custom = [
    "<user_1> кто идет на пары завтра?",
    "<user_2> я точно нет",
    "<user_3> а что завтра вообще?",
]
chat([{"role": "user", "content": c} for c in custom])

In [ ]:
print("=" * 60)
print("Test 3: Diversity check (5 generations, same context)")
print("=" * 60)
for i in range(5):
    print(f"\nGeneration {i+1}:")
    chat([{"role": "user", "content": c} for c in custom], max_new_tokens=64)

## 7. Export (Optional)

Merge LoRA into base model, optionally convert to GGUF (quantized), and push to HuggingFace Hub.

In [ ]:
# Merge LoRA weights into base model
merged_model = model.merge_and_unload()
merged_model.save_pretrained("merged_model_local")
tokenizer.save_pretrained("merged_model_local")
print("Merged model saved to merged_model_local/")

In [ ]:
HF_REPO = "vXofi/businessgpt-qwen3vl-2b-v3"
merged_model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Pushed merged 16-bit to https://huggingface.co/{HF_REPO}")

### GGUF Export (quantized)

Convert the merged model to GGUF Q4_K_M format using `llama.cpp`.
This builds `llama.cpp` from source (needs `cmake` and a C++ compiler — Xcode CLI tools on macOS).

In [ ]:
%%bash
# Build llama.cpp (one-time setup, takes ~2 min)
if [ ! -f llama.cpp/build/bin/llama-quantize ]; then
    git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
    cd llama.cpp && cmake -B build -DLLAMA_METAL=ON && cmake --build build -j
    echo "llama.cpp built successfully"
else
    echo "llama.cpp already built"
fi

In [ ]:
import subprocess, shutil

MERGED_DIR = "merged_model_local"
GGUF_DIR = "gguf_local"
os.makedirs(GGUF_DIR, exist_ok=True)

f16_gguf = os.path.join(GGUF_DIR, "model-f16.gguf")
q4_gguf = os.path.join(GGUF_DIR, "model-Q4_K_M.gguf")

# Step 1: Convert HF safetensors -> GGUF F16
print("Converting to GGUF F16...")
subprocess.run([
    "python", "llama.cpp/convert_hf_to_gguf.py",
    MERGED_DIR, "--outfile", f16_gguf, "--outtype", "f16"
], check=True)
print(f"  F16 GGUF: {os.path.getsize(f16_gguf)/1e9:.2f} GB")

# Step 2: Quantize F16 -> Q4_K_M
print("Quantizing to Q4_K_M...")
subprocess.run([
    "llama.cpp/build/bin/llama-quantize",
    f16_gguf, q4_gguf, "Q4_K_M"
], check=True)
print(f"  Q4_K_M GGUF: {os.path.getsize(q4_gguf)/1e9:.2f} GB")

# Clean up F16 intermediate file
os.remove(f16_gguf)
print(f"\nDone! Quantized model at: {q4_gguf}")

In [ ]:
# Push GGUF to HuggingFace Hub
from huggingface_hub import HfApi

GGUF_REPO = "vXofi/businessgpt-qwen3vl-2b-v3-q4_gguf"
api = HfApi()
api.create_repo(GGUF_REPO, exist_ok=True)
api.upload_file(
    path_or_fileobj=q4_gguf,
    path_in_repo="model-Q4_K_M.gguf",
    repo_id=GGUF_REPO,
)
print(f"Pushed GGUF to https://huggingface.co/{GGUF_REPO}")